# Stochastic transfer: does the learned rule survive noise?

The `origanchors` adapter localizes 32/32 on deterministic
`silent_break`. Section 6.2 attributes that to one rule: the broken pair
is the only thing that self-loops in period B, and nothing self-loops in
period A. This notebook tests that claim rather than asserting it.

Stochastic mode breaks the surface version of the rule. A healthy link
fails whenever its attempt misses, so self-loops appear everywhere. On
seed 7, period A has 13 pairs that self-loop at least once and period B
has 9, while only the target never succeeds. So two readings of the rule
come apart:

* **surface**: "the pair that self-loops in B" -> false under noise,
  should collapse
* **robust**: "the pair that never succeeds in B" -> still uniquely
  identifying, should hold

The evidence-only baselines bound it. "Uniquely dead in period B" scores
65/65 on stochastic `silent_break`; "biggest drop in success rate" scores
0.77. So the ceiling is unchanged at 1.00 and any collapse is the model,
not the task.

The adapter has never seen a stochastic world. Its anchors came from
`anchors.py`, which has no probabilities at all: a step either arrives or,
if the pair is broken, self-loops. So this is also the transfer test that
has been open since the original handoff.

## Either result is usable

* **Holds up** -> the model learned the robust rule, which survives a
  distribution it never trained on. That is a stronger claim than 6.2
  currently makes.
* **Collapses** -> it learned the surface form, the mechanism paragraph
  in 6.2 is right, and the fragility is quantified rather than argued.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, re, collections, subprocess
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG       = "sto"
ADAPTER_MATCH = "origanchors"
N_SEEDS       = 32
PROBES        = ["localization"]    # widen to None for all four
K             = 5

REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
# gen_payloads.py may sit in a different folder of the same dataset
# depending on how Kaggle nested the upload, so locate it separately
GEN_PATH = find_dir("gen_payloads.py")
_c = [h for h in glob.glob("/kaggle/input/**/adapter_config.json",
                           recursive=True) if ADAPTER_MATCH in h]
if len(_c) != 1:
    raise SystemExit(f"need one adapter matching {ADAPTER_MATCH!r}, got {_c}")
ADAPTER_PATH = os.path.dirname(_c[0])
OUT_DIR = "/kaggle/working"
print("repo:   ", REPO_PATH)
print("eval:   ", EVAL_PATH)
print("gen_payloads:", GEN_PATH)
print("adapter:", ADAPTER_PATH)

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel")
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8          # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"gpu: {torch.cuda.get_device_name(0)} | dtype: {DTYPE}")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
cfg = json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json")))
assert cfg["base_model_name_or_path"] == MODEL_NAME
prov = os.path.join(ADAPTER_PATH, "phase1_provenance.json")
if os.path.isfile(prov):
    p = json.load(open(prov))
    print("phase 1:", p.get("run"), "|", p.get("anchor_source"),
          "|", p["optimizer_steps"], "steps")

## Build the stochastic instances

Same code path as the payload files, with `deterministic=False`. They are
also written to disk so `baselines.py` can read them and print the
ceiling.

In [ ]:
sys.path.insert(0, EVAL_PATH)
sys.path.insert(0, GEN_PATH)
import gen_payloads as GP, ecpm_eval as E
rp, _ = GP.load_env(REPO_PATH)
E.attach(REPO_PATH)

PAY_DIR = f"{OUT_DIR}/payloads_silent_break_sto"
os.makedirs(PAY_DIR, exist_ok=True)
payloads, skipped = [], []
for seed in range(80):
    if len(payloads) >= N_SEEDS:
        break
    try:
        b = GP.build_for_seed(rp, seed, False, K, "silent_break")
    except (ValueError, RuntimeError):
        skipped.append(seed)
        continue
    json.dump(b, open(f"{PAY_DIR}/payload_seed{seed}_silent_break_sto_k{K}.json",
                      "w"))
    payloads.append(b)
print(f"{len(payloads)} stochastic instances: "
      f"{[p['seed'] for p in payloads]}")
print(f"ineligible: {skipped}")

# how badly is the surface rule broken in these instances?
TRIPLE = re.compile(r"\(([A-Z]), (a\d+), ([A-Z])\)")
loops_a, loops_b, unique_dead = [], [], 0
for p in payloads:
    ev = p["single"]["detection"]
    pre, post = ev.split("Observations, period A:", 1)[1] \
                  .split("Observations, period B:", 1)
    loops_a.append(len({(u, a) for u, a, v in TRIPLE.findall(pre) if u == v}))
    loops_b.append(len({(u, a) for u, a, v in TRIPLE.findall(post) if u == v}))
    ok = collections.Counter()
    seen = collections.Counter()
    for u, a, v in TRIPLE.findall(post):
        seen[(u, a)] += 1
        ok[(u, a)] += (u != v)
    dead = [q for q in seen if ok[q] == 0]
    unique_dead += (len(dead) == 1
                    and dead[0] == tuple(p["facts"]["target_pair"]))
print(f"\npairs self-looping at least once: period A median "
      f"{sorted(loops_a)[len(loops_a)//2]}, period B median "
      f"{sorted(loops_b)[len(loops_b)//2]}")
print(f"target is the uniquely never-succeeding pair: {unique_dead}/{len(payloads)}")
print("-> the surface rule is false here; the robust one still identifies it")

## The ceiling on these instances

In [ ]:
print(subprocess.run([sys.executable,
                      os.path.join(EVAL_PATH, "baselines.py"), PAY_DIR],
                     capture_output=True, text=True).stdout)

## Model

`arm_c` runs with the adapter, `icl` inside `disable_adapter()`, so the
only difference is phase 1.

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map={"": 0}),
    ADAPTER_PATH)
model.eval()
model.config.use_cache = True

def generate(messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    e = {k: v.to(model.device) for k, v in e.items()}
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

def make_ask(use_adapter):
    def ask(messages, n):
        if use_adapter:
            return generate(messages, n)
        with model.disable_adapter():
            return generate(messages, n)
    return ask

ASK = {"arm_c": make_ask(True), "icl": make_ask(False)}
print("loaded | GPU GB:", round(torch.cuda.memory_allocated() / 1e9, 2))

## Run

In [ ]:
rows = []
for arm in ("arm_c", "icl"):
    t0 = time.time()
    rows += E.run_arm(payloads, ASK[arm], arm=arm, mode="single",
                      probes=PROBES,
                      out_path=f"{OUT_DIR}/raw_{RUN_TAG}_{arm}.jsonl")
    print(f"  {arm} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
import pandas as pd

print(f"{'arm':8} {'exact':>10} {'node-level':>12}")
res = {}
for arm in ("arm_c", "icl"):
    loc = [r for r in rows if r["arm"] == arm and r["probe"] == "localization"
           and r["scored"].get("applicable")]
    ex = [r["seed"] for r in loc if r["scored"].get("correct")]
    nd = sum(1 for r in loc if r["parsed"].get("node") == r["target"][0])
    res[arm] = ex
    print(f"{arm:8} {len(ex):>4}/{len(loc):<5} {nd:>6}/{len(loc):<5}")

print("\nfor comparison, deterministic silent_break on the same adapter:")
print("  arm_c 32/32, icl 4/32, evidence-only rule 32/32")
print("stochastic evidence-only baselines: uniquely-dead 65/65, "
      "biggest-drop 0.77")

print(f"\narm_c correct on: {sorted(res['arm_c'])}")
print(f"icl correct on:   {sorted(res['icl'])}")
print("\nMcNemar arm_c vs icl:",
      E.mcnemar([r for r in rows if r["arm"] == "arm_c"],
                [r for r in rows if r["arm"] == "icl"]))

df = pd.DataFrame([{"seed": r["seed"], "gold": " ".join(r["target"]),
                    "arm_c": next((f"{x['parsed'].get('node')} "
                                   f"{x['parsed'].get('action')}"
                                   for x in rows if x["seed"] == r["seed"]
                                   and x["arm"] == "arm_c"), "-"),
                    "icl": next((f"{x['parsed'].get('node')} "
                                 f"{x['parsed'].get('action')}"
                                 for x in rows if x["seed"] == r["seed"]
                                 and x["arm"] == "icl"), "-")}
                   for r in rows if r["arm"] == "arm_c"
                   and r["probe"] == "localization"])
display(df.sort_values("seed"))

## Which rule did it learn?

If `arm_c` is high, it is tracking "never succeeds in period B", which
survives noise. If it is low, it was tracking "self-loops in period B",
which stochastic mode makes useless. The check below asks whether its
wrong answers land on pairs that do self-loop in B, which is the
signature of the surface rule misfiring.

In [ ]:
hit_selfloop = hit_dead = wrong = 0
for r in rows:
    if r["arm"] != "arm_c" or r["probe"] != "localization":
        continue
    if r["scored"].get("correct") or r["parsed"]["status"] != "ok":
        continue
    pay = next(p for p in payloads if p["seed"] == r["seed"])
    post = pay["single"]["detection"].split("Observations, period B:", 1)[1]
    said = (r["parsed"]["node"], r["parsed"]["action"])
    loops = {(u, a) for u, a, v in TRIPLE.findall(post) if u == v}
    ok = collections.Counter()
    seen = collections.Counter()
    for u, a, v in TRIPLE.findall(post):
        seen[(u, a)] += 1
        ok[(u, a)] += (u != v)
    wrong += 1
    hit_selfloop += said in loops
    hit_dead += said in seen and ok[said] == 0
if wrong:
    print(f"of {wrong} wrong arm_c answers:")
    print(f"  {hit_selfloop} named a pair that self-loops at least once in B")
    print(f"  {hit_dead} named a pair that never succeeds in B")
    print("\na high first number is the surface rule misfiring under noise")
else:
    print("no wrong answers to characterise")

## Package

In [ ]:
import zipfile
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if (not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path
                or ".ipynb_checkpoints" in path):
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")